In [1]:
import numpy as np
from math import log10, sqrt 
import scipy.misc
import struct
import time
from datetime import datetime
import matplotlib.pyplot as plt
from skimage import measure
from pynq import Overlay
from pynq import MMIO
from scipy.fftpack import dct, idct
import csv
import os

def dct2(block):
    return dct(dct(block.T).T)

def idct2(block):
    return idct(idct(block.T).T)

def rgb2gray(rgb):
    return np.dot(rgb[...,:3], [0.2989, 0.5870, 0.1140])
def ZigZag(a,r):
    n=0
    j=0
    i=0
    [M,N] = a.shape
    b=np.zeros((M,M))
    for s in range(r):
        b[j,i]=a[j,i]
        if (s<((M**2+M)/2)-1):            
            if ((n%2)==0):  
                if (i<n):
                    i=i+1
                    j=j-1
                else: 
                    n=n+1
                    i=n
            else:
                if (j<n):
                    j=j+1
                    i=i-1
                else:
                    n=n+1
                    j=n
        elif (s==((M**2+M)/2)-1):
                n=n+1
                i=i+1
        else:
            
            if ((n%2)==0):  
                if (i<M-1):
                    i=i+1
                    j=j-1
                else:
                    n=n+1
                    i=M-1
                    j=n-(M-1)
            else:
                if (j<M-1):
                    j=j+1
                    i=i-1
                else:
                    n=n+1
                    j=M-1
                    i=n-(M-1)
    return b
def PSNR(original, compressed): 
    mse = np.mean((original - compressed) ** 2) 
    if(mse == 0):  # MSE is zero means no noise is present in the signal . 
                  # Therefore PSNR have no importance. 
        return 100
    max_pixel = 255.0
    psnr = 20 * log10(max_pixel / sqrt(mse)) 
    return psnr
def inverseDCT(A,C0,S):
    A=np.dot(S,A)
    I1=np.dot(np.transpose(C0),A)
    I1=np.dot(I1,S)
    B=np.dot(np.transpose(C0),np.transpose(I1))
    return B
def writeDocument(matriz):
    with open(filename, 'w',newline='') as f:
        w = csv.writer(f)
        for row in matriz:
            if not len(row)==0:
                w.writerow(row)
        f.close()
#==== Seleccionar y Cargar Imagen
def menuBit():
    os.system('cls') # NOTA para windows cambiar clear por cls
    print ("Selecciona Aproximación a Cargar")
    print ("\t1 - bas08")
    print ("\t2 - bas11_05")
    print ("\t3 - bas11_1")
    print ("\t4 - bas11_0")
    print ("\t5 - cb11")
    print ("\t6 - cb11mod")
    print ("\t7 - brahimi")
    print ("\t8 - hawell")
    print ("\t9 - potluri")
    print ("\t10 - senapati")
    print ("\t11 - tablada")


In [3]:
#==== Seleccionar y Cargar Aproximación (Archivo .bit y Matriz de Conversión)
OpcB = 0;
while (OpcB==0):
    menuBit()
    OpcB = input("Opcion >> ")
    if OpcB =="1":
        over = "DCT8x8_BAS2008.bit"
        filename = 'Bas08.csv'
        C0=np.array([[1,1,1,1,1,1,1,1],    
            [1,1,0,0,0,0,-1,-1],
            [1,0.5,-0.5,-1,-1,-0.5,0.5,1],
            [0,0,-1,0,0,1,0,0],
            [1,-1,-1,1,1,-1,-1,1],
            [1,-1,0,0,0,0,1,-1],
            [0.5,-1,1,-0.5,-0.5,1,-1,0.5],
            [0,0,0,-1,1,0,0,0]])
        S=abs(np.sqrt(np.linalg.inv(np.dot(C0,np.transpose(C0)))))
        S=np.diag(np.diagonal(S,0,0,1))
    elif OpcB =="2":
        over = "DCT8x8_BAS11_05.bit"
        filename = 'Bas11_05.csv'
        C0=np.array([[1,1,1,1,1,1,1,1],    
            [1,1,0,0,0,0,-1,-1],
            [1,0.5,-0.5,-1,-1,-0.5,0.5,1],
            [0,0,1,0,0,-1,0,0],
            [1,-1,-1,1,1,-1,-1,1],
            [0,0,0,1,-1,0,0,0],
            [1,-1,0,0,0,0,1,-1],
            [0.5,-1,1,-0.5,-0.5,1,-1,0.5]])
        S=abs(np.sqrt(np.linalg.inv(np.dot(C0,np.transpose(C0)))))
        S=np.diag(np.diagonal(S,0,0,1))
    elif OpcB =="3":
        over = "DCT8x8_BAS11_1.bit"
        filename = 'Bas11_1.csv'
        C0=np.array([[1,1,1,1,1,1,1,1],    
            [1,1,0,0,0,0,-1,-1],
            [1,1,-1,-1,-1,-1,1,1],
            [0,0,1,0,0,-1,0,0],
            [1,-1,-1,1,1,-1,-1,1],
            [0,0,0,1,-1,0,0,0],
            [1,-1,0,0,0,0,1,-1],
            [1,-1,1,-1,-1,1,-1,1]])
        S=abs(np.sqrt(np.linalg.inv(np.dot(C0,np.transpose(C0)))))
        S=np.diag(np.diagonal(S,0,0,1))
    elif OpcB =="4":
        over = "DCT8x8_BAS11_0.bit"
        filename = 'Bas11_0.csv'
        C0=np.array([[1,1,1,1,1,1,1,1],    
            [1,1,0,0,0,0,-1,-1],
            [0,0,1,0,0,-1,0,0],
            [0,0,0,1,-1,0,0,0],
            [1,0,0,-1,-1,0,0,1],
            [0,1,-1,0,0,-1,1,0],
            [1,-1,0,0,0,0,1,-1],
            [1,-1,-1,1,1,-1,-1,1]])
        S=abs(np.sqrt(np.linalg.inv(np.dot(C0,np.transpose(C0)))))
        S=np.diag(np.diagonal(S,0,0,1))
    elif OpcB =="5":
        over = "DCT8x8_cb2011.bit"
        filename = 'cb2011.csv'
        C0=np.array([[1,1,1,1,1,1,1,1],    
            [1,1,1,0,0,-1,-1,-1],
            [1,0,0,-1,-1,0,0,1],
            [1,0,-1,-1,1,1,0,-1],
            [1,-1,-1,1,1,-1,-1,1],
            [1,-1,0,1,-1,0,1,-1],
            [0,-1,1,0,0,1,-1,0],
            [0,-1,1,-1,1,-1,1,0]])
        S=abs(np.sqrt(np.linalg.inv(np.dot(C0,np.transpose(C0)))))
        S=np.diag(np.diagonal(S,0,0,1))
    elif OpcB =="6":
        over = "DCT8x8_Bayer2012.bit"
        filename = 'cb2011mod.csv'
        C0=np.array([[1,1,1,1,1,1,1,1],    
            [1,0,0,0,0,0,0,-1],
            [1,0,0,-1,-1,0,0,1],
            [0,0,-1,0,0,1,0,0],
            [1,-1,-1,1,1,-1,-1,1],
            [0,-1,0,0,0,0,1,0],
            [0,-1,1,0,0,1,-1,0],
            [0,0,0,-1,1,0,0,0]])
        S=abs(np.sqrt(np.linalg.inv(np.dot(C0,np.transpose(C0)))))
        S=np.diag(np.diagonal(S,0,0,1))
    elif OpcB =="7":
        over = "DCT8x8_Brahimi.bit"
        filename = 'Brahimi.csv'
        C0=np.array([[1,1,1,1,1,1,1,1],    
            [1,1,0,0,0,0,-1,-1],
            [1,0,0,-1,-1,0,0,1],
            [0,0,-1,0,0,1,0,0],
            [1,-1,-1,1,1,-1,-1,1],
            [1,-1,0,0,0,0,1,-1],
            [0,-1,1,0,0,1,-1,0],
            [0,0,0,-1,1,0,0,0]])
        S=abs(np.sqrt(np.linalg.inv(np.dot(C0,np.transpose(C0)))))
        S=np.diag(np.diagonal(S,0,0,1))
    elif OpcB =="8":
        over = "DCT8x8_PADCT.bit" #hawell
        filename = 'Haweel16.csv'
        C0=np.array([[1,1,1,1,1,1,1,1],    
            [1,1,0,0,0,0,-1,-1],
            [1,1,-1,-1,-1,-1,1,1],
            [0,0,-1,0,0,1,0,0],
            [1,-1,-1,1,1,-1,-1,1],
            [1,-1,0,0,0,0,1,-1],
            [1,0,0,-1,-1,0,0,1],
            [0,0,0,-1,1,0,0,0]])
        #S=abs(np.sqrt(np.linalg.inv(np.dot(C0,np.transpose(C0)))))
        S=np.zeros((8,8))
        S[0,0]=1/sqrt(8)
        S[1,1]=1/2
        S[2,2]=1/sqrt(8)
        S[3,3]=1/sqrt(2)
        S[4,4]=1/sqrt(8)
        S[5,5]=1/2
        S[6,6]=1/2
        S[7,7]=1/sqrt(2)

    elif OpcB =="9":
        over = "DCT8x8_14sum.bit" #potluri
        filename = 'Potluri.csv'
        C0=np.array([[1,1,1,1,1,1,1,1],    
            [0,1,0,0,0,0,-1,0],
            [1,0,0,-1,-1,0,0,1],
            [1,0,0,0,0,0,0,-1],
            [1,-1,-1,1,1,-1,-1,1],
            [0,0,0,1,-1,0,0,0],
            [0,-1,1,0,0,1,-1,0],
            [0,0,1,0,0,-1,0,0]])
        S=abs(np.sqrt(np.linalg.inv(np.dot(C0,np.transpose(C0)))))
        S=np.diag(np.diagonal(S,0,0,1))
    elif OpcB =="10":
        over = "DCT8x8_Senapati2010.bit"
        filename = 'Senapati.csv'
        C0=np.array([[1,1,1,1,1,1,1,1],    
            [1,1,0,0,0,0,-1,-1],
            [1,0.5,-0.5,-1,-1,-0.5,0.5,1],
            [0,0,-1,0,0,1,0,0],
            [1,-1,-1,1,1,-1,-1,1],
            [1,-1,0,0,0,0,1,-1],
            [0.5,0,0,-0.5,-0.5,0,0,0.5],
            [0,0,0,-1,1,0,0,0]])
        #S=abs(np.sqrt(np.linalg.inv(np.dot(C0,np.transpose(C0)))))
        S=np.zeros((8,8))
        S[0,0]=1
        S[1,1]=sqrt(2)
        S[2,2]=2*sqrt(2/5)
        S[3,3]=2
        S[4,4]=1
        S[5,5]=sqrt(2)
        S[6,6]=2*sqrt(2/5)
        S[7,7]=2
        S=S/(2*sqrt(2))
 
    elif OpcB =="11":
        over = "DCT8x8_Tablada2017.bit" 
        filename = 'Tablada.csv'
        C0=np.array([[1,1,1,1,1,1,1,1],    
            [1,1,1,0,0,-1,-1,-1],
            [1,0,0,-1,-1,0,0,1],
            [1,0,-2,-1,1,2,0,-1],
            [1,-1,-1,1,1,-1,-1,1],
            [1,-2,0,1,-1,0,2,-1],
            [0,-1,1,0,0,1,-1,0],
            [0,-1,1,-1,1,-1,1,0]])
        #S=abs(np.sqrt(np.linalg.inv(np.dot(C0,np.transpose(C0)))))
        S=np.zeros((8,8))
        S[0,0]=1/sqrt(8)
        S[1,1]=1/sqrt(6)
        S[2,2]=1/2
        S[3,3]=1/sqrt(12)
        S[4,4]=1/sqrt(8)
        S[5,5]=1/sqrt(12)
        S[6,6]=1/2
        S[7,7]=1/sqrt(6)
    else:
        OpcB = 0
        break
        print ("")
        input("Opción incorrecta...\npulsa 1 tecla para continuar")

OL = Overlay(over) #==== cambiar el bit y tcl

OL.download()
OL.ip_dict

myip = MMIO(0x43C00000,0xFFFF)


M=8
C = np.zeros((M,M))
im_rec=np.zeros((X,Y))
im_dct=np.zeros((X,Y))

input_registers = myip.array[0:32]
output_registers = myip.array[32:64]
res_temp=np.uint32(np.zeros((32)))
startTime = datetime.now()     #======== Inicia la transformación DCT de toda la imagen ========
escribir=np.zeros((64,25))
escribir2=np.zeros((64,25))
escribir3=np.zeros((64,25))
for im in range(25):
    image = plt.imread(str(im+1)+".tiff")
    [X, Y]=image.shape
    im_rec=np.zeros((X,Y))
    im_dct=np.zeros((X,Y))
    for i in range(int(X/8)):      #<== 256x256 ==> 32x32x8 = 8,192 TransAXI; 512x512 ==> 64x64x8 = 32,768 TransAXI
        for j in range(int(Y/8)):  #<== 256x256 ==> 32 repeticiones;  512x512 ==> 64 repeticiones
            matriz2=np.zeros((M,M))
            matriz3=np.zeros((M,M))
            C = np.zeros((M,M))
            A=image[(i*8):(i*8)+8,(j*8):(j*8)+8]
            B=np.transpose(A).ravel()
            input_data=np.zeros((32))
            for k in range(32):
                input_data[k]=(B[k*2]*65536)+B[(k*2)+1]

            input_registers[:] = np.uint32(input_data)

            res_temp[:]=output_registers
            for m in range(M):  #<==== Recibe 8 datos de 16 bits ==> 128 bits en 4 transacciones
                #res_temp=myip.read((16*m)+128); #<== Ojo: Recibe 2 datos de 16 bits => Verificar si se puede recibir 4 de 8
                #res_temp[:]=output_registers[m*4:(m*4)+4]
                C[1,m] = int(struct.unpack('h', struct.pack('H', int(res_temp[(m*4)]/65536)))[0])  #H unsigned short, h signed short
                C[0,m] = int(struct.unpack('h', struct.pack('H', int((res_temp[(m*4)]&0xFFFF))))[0])
                #res_temp=myip.read((16*m)+132)
                C[3,m] = int(struct.unpack('h', struct.pack('H', int(res_temp[(m*4)+1]/65536)))[0])
                C[2,m] = int(struct.unpack('h', struct.pack('H', int((res_temp[(m*4)+1]&0xFFFF))))[0])
                #res_temp=myip.read((16*m)+136)
                C[5,m] = int(struct.unpack('h', struct.pack('H', int(res_temp[(m*4)+2]/65536)))[0])
                C[4,m] = int(struct.unpack('h', struct.pack('H', int((res_temp[(m*4)+2]&0xFFFF))))[0])
                #res_temp=myip.read((16*m)+140)
                C[7,m] = int(struct.unpack('h', struct.pack('H', int(res_temp[(m*4)+3]/65536)))[0])
                C[6,m] = int(struct.unpack('h', struct.pack('H', int((res_temp[(m*4)+3]&0xFFFF))))[0])
            C = np.dot(np.transpose(np.dot(C,S)),S); #<=== Ortogonalización de Aproximación
            C = np.transpose(C)
            im_dct[(i*8):(i*8)+8,(j*8):(j*8)+8] = C
    print (datetime.now()-startTime)

    #======== Inicia la transformación inversa IDCT de toda la imagen despreciando coeficientes de DCT ========
    for r in range(64):   #<==== 256x256 ==> 8,192x32 = 252,144 TransAXI;  512x512 ==> 32,768x32 = 1'048,576 TransAXI
        for i in range(int(X/8)):      #<== 256x256 ==> 32x32x8 = 8,192 TransAXI; 512x512 ==> 64x64x8 = 32,768 TransAXI
            for j in range(int(Y/8)):  #<== 256x256 ==> 32 repeticiones;  512x512 ==> 64 repeticiones
                C = im_dct[(i*8):(i*8)+8,(j*8):(j*8)+8]    
                C = ZigZag(C,r+1)
                im_rec[(i*8):(i*8)+8,(j*8):(j*8)+8] = inverseDCT(C,C0,S)

        im_rec[im_rec > 255] = 255; #<=== Aquí solo tomas 8 bits despreciando lo demás,
        im_rec[im_rec < 0] = 0
        escribir[r,im]=PSNR(image,im_rec)
        MSE = np.square(np.subtract(image,im_rec)).mean()
        escribir2[r,im]=MSE
        #print(MSE)
        s = measure.compare_ssim(image, np.uint8(im_rec))
        escribir3[r,im]=s
promedios=np.zeros((64,3))
for r in range(64):
    promedios[r,0]=np.mean(escribir[r,:])
    promedios[r,1]=np.mean(escribir2[r,:])
    promedios[r,2]=np.mean(escribir3[r,:])

with open(filename, 'w',newline='') as f:
        w = csv.writer(f)
        for row in promedios:
            if not len(row)==0:
                w.writerow(row)
        f.close()

Selecciona Aproximación a Cargar
	1 - bas08
	2 - bas11_05
	3 - bas11_1
	4 - bas11_0
	5 - cb11
	6 - cb11mod
	7 - brahimi
	8 - hawell
	9 - potluri
	10 - senapati
	11 - tablada
Opcion >> 1


/usr/local/lib/python3.6/dist-packages/pynq/pl_server/device.py:594: UserWarning: Users will not get PARAMETERS / REGISTERS information through TCL files. HWH file is recommended.
  warnings.warn(message, UserWarning)


0:00:12.528562
0:01:35.910103
0:07:17.348096
0:21:37.307198
0:23:01.279097
0:28:43.561547
0:43:28.854194
0:47:35.355823
0:51:41.713588
0:55:48.590472
0:59:29.221376
1:00:29.078062
1:01:53.150843
1:05:58.312326
1:11:40.575907
1:26:01.016335
1:27:01.003771
1:28:25.364483
1:32:31.206550
1:36:38.147677
1:40:43.252964
1:44:48.472319
1:48:54.092235
1:52:59.923774
1:57:05.362911
